In [67]:
%pip install langchain

Note: you may need to restart the kernel to use updated packages.


In [68]:
%pip install langchain-openai

Note: you may need to restart the kernel to use updated packages.


In [69]:
%pip install langsmith

Note: you may need to restart the kernel to use updated packages.


In [70]:
%pip install langsmith openai

Note: you may need to restart the kernel to use updated packages.


In [71]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [72]:
import os
from dotenv import load_dotenv

# Load environment variables (override=True ensures .env changes are always picked up)
load_dotenv(override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Verify Adzuna keys loaded correctly
print("ADZUNA_APP_ID:", os.getenv("ADZUNA_APP_ID"))
print("ADZUNA_API_KEY:", "set" if os.getenv("ADZUNA_API_KEY") else "NOT SET")

ADZUNA_APP_ID: dbd7085d
ADZUNA_API_KEY: set


In [73]:
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage, AIMessage, SystemMessage

# Initialize a chat model
model = ChatOpenAI(model="gpt-5-nano", temperature=0.7, api_key=OPENAI_API_KEY)

# Create message objects
system_msg = SystemMessage(content="You are a helpful assistant.")
human_msg = HumanMessage(content="Hello, how are you?")

# Pass messages to the model
messages = [system_msg, human_msg]
response = model.invoke(messages)

print(f"Type of response: {type(response)}")
print(f"\nResponse content: {response.content}")

Type of response: <class 'langchain_core.messages.ai.AIMessage'>

Response content: Hi there! I’m here and ready to help. How can I assist you today?


Adzuna tool

In [ ]:
import os
import requests
from typing import Optional, Literal

from langchain.tools import tool

app_id = os.getenv("ADZUNA_APP_ID")
app_key = os.getenv("ADZUNA_APP_KEY")

if not app_id or not app_key:
    raise ValueError("Error: ADZUNA_APP_ID or ADZUNA_APP_KEY not set.")

@tool(
    "adzuna_search",
    parse_docstring=True,
    description=(
        "Search for job listings on Adzuna."
        "Use this tool to find jobs based on title, location, and salary range."
    ),
)
def adzuna_jobs(
    what: str,
    country: Optional[str] = "us",
    where: Optional[str] = None,
    results_per_page: int = 10,
    salary_min: Optional[int] = None,
    salary_max: Optional[int] = None,
    full_time: Optional[Literal["1"]] = None,
    part_time: Optional[Literal["1"]] = None,
    contract: Optional[Literal["1"]] = None,
    permanent: Optional[Literal["1"]] = None,
    sort_by: Optional[str] = None,
    max_days_old: Optional[int] = None,
    category: Optional[str] = None,
    company: Optional[str] = None,
    distance: Optional[int] = None,
    what_exclude: Optional[str] = None,
    title_only: Optional[str] = None,
) -> dict:
    """
    Search for job listings on Adzuna.

    Args:
        what: Job title or keywords to search for (e.g., 'data scientist').
        country: Two-letter country code (e.g., 'us', 'gb', 'ca'). Defaults to 'us'.
        where: City or region to filter by (e.g., 'New York', 'Charlotte').
        results_per_page: Number of results to return (1-50). Defaults to 10.
        salary_min: Minimum annual salary filter.
        salary_max: Maximum annual salary filter.
        full_time: Set to 1 to return only full-time positions.
        part_time: Set to 1 to return only part-time positions.
        contract: Set to 1 to return only contract positions.
        permanent: Set to 1 to return only permanent positions.
        sort_by: How to sort results. One of 'default', 'hybrid', 'date', 'salary', 'relevance'.
        max_days_old: Only return jobs posted within this many days.
        category: Adzuna category tag (e.g., 'it-jobs', 'healthcare-nursing-jobs', 'engineering-jobs', 'accounting-finance-jobs').
        company: Filter by company name.
        distance: Radius in km from the location to search within.
        what_exclude: Keywords to exclude from results (e.g., 'intern junior').
        title_only: Keywords that must appear in the job title only.
    
    Returns:
        A dictionary with job listings or an error message.
    
    Raises:
        ValueError: If no job listings are found or if there is an error with the API request.
    """
    try:
        print(f"Calling Adzuna API with query='{what}'")

        #This is the base URL for the API. However, when filtering for countries in different databases and requires URL append. 
        url = f"https://api.adzuna.com/v1/api/jobs/{country}/search/1"

        #We add some parameters that the LLM can't control and are required for the API call.
        params = {
            "app_id": app_id,
            "app_key": app_key,
            "content-type": "application/json",       
        }

        func_args = {
            "what": what,
            "where": where,
            "results_per_page": results_per_page,
            "salary_min": salary_min,
            "salary_max": salary_max,
            "full_time": full_time,
            "part_time": part_time,
            "contract": contract,
            "permanent": permanent,
            "sort_by": sort_by,
            "max_days_old": max_days_old,
            "category": category,
            "company": company,
            "distance": distance,
            "what_exclude": what_exclude,
            "title_only": title_only,
        }

        params.update({k: v for k, v in func_args.items() if v is not None})

        response = requests.get(url, params=params, timeout=10)
        
        if response.status_code == 400:
            return {
                "success": False,
                "error": "Bad request to Adzuna. You may have used an invalid parameter combination. Try simplifying your search.",
            }
        elif response.status_code == 401:
            return {
                "success": False,
                "error": "Authentication failed with Adzuna API. Check API credentials.",
            }
        elif response.status_code >= 500:
            return {
                "success": False,
                "error": "Adzuna API is temporarily unavailable. Try again in a moment.",
            }
        elif response.status_code != 200:
            return {
                "success": False,
                "error": f"Adzuna API error {response.status_code}: {response.text}",
            }
        
        data = response.json()
        jobs = data.get("results", [])

        if not jobs:
            return {
                "success": False,
                "error": f"No jobs found for '{what}'. Try different keywords, remove filters, or search a different location.",
            }
        
        return {
            "success": True,
            "data": jobs,
            "count": data.get("count", len(jobs)),
        }
        
    except requests.exceptions.Timeout:
        return {
            "success": False,
            "error": "Request to Adzuna timed out. Try again with a simpler query.",
        }
    except requests.exceptions.RequestException as e:
        return {
            "success": False,
            "error": f"Network error calling Adzuna: {str(e)}. Check your connection.",
        }
    except Exception as e:
        return {
            "success": False,
            "error": f"Unexpected error: {str(e)}",
        }

Muse Tool

In [ ]:
import os
import requests
from typing import Optional, Literal

from langchain.tools import tool

api_key = os.getenv("MUSE_API_KEY")

if not api_key:
    raise ValueError("Error: MUSE_API_KEY not set.")

@tool(
    "muse_search",
    parse_docstring=True,
    description=(
        "Search for job listings on The Muse."
        "Use this tool to find jobs by category, location, experience level, or company."
    ),
)
def muse_jobs(
    category: Optional[str] = None,
    location: Optional[str] = None,
    level: Optional[str] = None,
    company: Optional[str] = None,
    page: int = 0,
    descending: Optional[bool] = None,
) -> dict:
    """
    Search for job listings on The Muse.

    Args:
        category: Job category to filter by (e.g., 'Software Engineering', 'Data Science', 'Marketing', 'Sales', 'Design', 'Education', 'Healthcare', 'Finance').
        location: City and state to filter by (e.g., 'New York, NY', 'San Francisco, CA', 'Chicago, IL', 'Flexible / Remote').
        level: Experience level filter. One of 'Internship', 'Entry Level', 'Mid Level', 'Senior Level', 'Management'.
        company: Company name to filter by (e.g., 'Meta', 'Ansys').
        page: Page number for results (0-indexed). Defaults to 0.
        descending: Set to true to sort results in descending order.
    
    Returns:
        A dictionary with job listings or an error message.
    
    Raises:
        ValueError: If no job listings are found or if there is an error with the API request.
    """
    try:
        print(f"Calling The Muse API with category='{category}', location='{location}', level='{level}', company='{company}'")

        #This is the base URL for the API. However, when filtering for countries in different databases and requires URL append. 
        url = "https://www.themuse.com/api/public/jobs"

        #We add some parameters that the LLM can't control and are required for the API call.
        params = {
            "api_key": api_key,
        }

        #Build the params for the http/api rcall and filter out none values.
        func_args = {
            "category": category,
            "location": location,
            "level": level,
            "company": company,
            "page": page,
            "descending": descending,
        }

        params.update({k: v for k, v in func_args.items() if v is not None})
        
        #Call the API and return results or error message
        response = requests.get(url, params=params, timeout=10)

        if response.status_code == 401:
            return {
                "success": False,
                "error": "Authentication failed with Muse API. Check API key.",
            }
        elif response.status_code >= 500:
            return {
                "success": False,
                "error": "Muse API is temporarily unavailable. Try again later.",
            }
        elif response.status_code != 200:
            return {
                "success": False,
                "error": f"Muse API error {response.status_code}: {response.text}",
            }
        
        data = response.json()
        jobs = data.get("results", [])
        
        if not jobs:
            return {
                "success": False,
                "error": f"No jobs found with those filters. Try broadening your search (e.g., different location, remove level filter, or try a different category).",
            }
        
        return {
            "success": True,
            "data": jobs,
            "total": data.get("total"),
            "page": data.get("page"),
            "page_count": data.get("page_count"),
        }
        
    except requests.exceptions.Timeout:
        return {
            "success": False,
            "error": "Request to Muse timed out. Try a simpler search.",
        }
    except requests.exceptions.RequestException as e:
        return {
            "success": False,
            "error": f"Network error calling Muse: {str(e)}",
        }
    except Exception as e:
        return {
            "success": False,
            "error": f"Unexpected error: {str(e)}",
        }



In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="openai:gpt-5-nano",
    tools=[adzuna_jobs, muse_jobs],
    system_prompt="You are a helpful job search assistant with two tools: " \
    "1)Adzuna: Best for keyword searches, salary filtering, and employment type (full-time, contract, etc.) " \
    "2)The Muse: Best for browsing by job category, experience level, or specific company."\
    "You can use both tools to broaden the scope and find more jobs, a single tool to narrow down results, or none."


In [77]:
import os
result = agent.invoke(
    {"messages": [{"role": "user", "content": "find me sales jobs in san francisco with a salary above 100k"}]}
)
print(result["messages"][-1].content)

Calling Adzuna API with query='sales'
Here are Salesforce/sales roles in San Francisco with salaries above $100k (based on current Adzuna results). For each, I’ve noted a rough salary range when available and whether the role is full-time.

- Principle ProServe Account Executive, NAMER Rtl/Cpg/Manu — Amazon
  - Location: SoMa, San Francisco
  - Salary: from about $520,792+ (max not stated)
  - Type: Full-time
  - Link: https://www.adzuna.com/land/ad/5612422107?se=0pqRUQUN8RGSJfyON74cmg&utm_medium=api&utm_source=dbd7085d&v=C31A39F8D54F3E2595C135C44B939D55D0B6E098

- RVP, Strategic Sales — Decagon
  - Location: San Francisco, CA
  - Salary: $520,000–$600,000
  - Type: Full-time
  - Link: https://www.adzuna.com/details/5624325420?utm_medium=api&utm_source=dbd7085d

- Director, Strategic Enterprise Sales (Bay Area) — Zendesk
  - Location: San Francisco, CA
  - Salary: $442,000–$662,000
  - Type: Full-time
  - Link: https://www.adzuna.com/details/5625891613?utm_medium=api&utm_source=dbd7085

In [1]:
import os
from dotenv import load_dotenv

load_dotenv("../.env", override=True)  # adjust path if needed
print("key loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("key prefix:", os.getenv("OPENAI_API_KEY", "")[:7])  # should look like sk-...


key loaded: True
key prefix: sk-proj


In [2]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(api_key=os.getenv("OPENAI_API_KEY"), model="gpt-5-nano")


In [3]:
resp = llm.invoke("Write a 3-bullet professional summary for a data analyst resume.")
print(resp.content)


- Data-driven analyst translating complex data into actionable business insights using SQL, Python (pandas), and Tableau/Power BI.
- Proficient in data wrangling, ETL, statistical analysis, and building scalable dashboards to track KPIs and support data-driven decision making.
- Collaborative communicator who partners with stakeholders to define questions, design analyses, and present clear, concise findings to non-technical audiences.


In [4]:
test_prompt = """
Name: Alex Kim
Experience: 2 years SQL, Python, Tableau
Goal: Data Analyst role in healthcare
Return exactly 3 resume summary bullets.
"""
resp = llm.invoke(test_prompt)
print(resp.content)


- Data Analyst with 2 years of experience leveraging SQL, Python (pandas, NumPy), and Tableau to build dashboards, automate reports, and derive insights that inform healthcare operations and patient care strategies.
- Proficient in translating clinical and administrative requirements into actionable analytics, performing data cleaning/ETL, cohort analyses, and KPI tracking to drive operational efficiency and patient outcomes.
- Strong communicator who collaborates with cross-functional teams to present complex findings to non-technical stakeholders, maintain data quality, and deliver scalable analytics solutions in healthcare environments.
